# 01 — Bronze Layer: Raw Ingestion

**Goal:** land the raw Pokémon datasets into the Bronze layer with an explicit schema and ingestion metadata — **no business transformations** happen here. That's Silver's job.

**Sources:**
- `pokemon.csv` — ~800 Pokémon with base stats, type(s), generation, legendary flag
- `combats.csv` — ~50,000 simulated 1v1 battles (original dataset, small-scale)

**Output:** Parquet files in `data/bronze/`, schema-enforced, with audit columns added.

In [1]:
import findspark
findspark.init()

from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import StructType, StructField, IntegerType, StringType, BooleanType

spark = (
    SparkSession.builder
    .appName("PokemonBronzeIngest")
    .master("local[*]")
    .getOrCreate()
)

spark.sparkContext.setLogLevel("WARN")
spark

## Explicit schemas

We define schemas explicitly instead of using `inferSchema=True`. This is a deliberate engineering choice: schema inference forces Spark to do an extra full read pass over the data to guess types, which doesn't scale well and can silently guess wrong (e.g. reading `Legendary` as string instead of boolean). Explicit schemas are faster, deterministic, and self-documenting.

In [2]:
pokemon_schema = StructType([
    StructField("#", IntegerType(), True),
    StructField("Name", StringType(), True),
    StructField("Type 1", StringType(), True),
    StructField("Type 2", StringType(), True),
    StructField("HP", IntegerType(), True),
    StructField("Attack", IntegerType(), True),
    StructField("Defense", IntegerType(), True),
    StructField("Sp. Atk", IntegerType(), True),
    StructField("Sp. Def", IntegerType(), True),
    StructField("Speed", IntegerType(), True),
    StructField("Generation", IntegerType(), True),
    StructField("Legendary", BooleanType(), True),
])

combats_schema = StructType([
    StructField("First_pokemon", IntegerType(), True),
    StructField("Second_pokemon", IntegerType(), True),
    StructField("Winner", IntegerType(), True),
])

## Read raw CSVs

In [3]:
BRONZE_PATH = "../data/bronze"

df_pokemon_raw = spark.read.csv(
    f"{BRONZE_PATH}/pokemon.csv",
    header=True,
    schema=pokemon_schema,
)

df_combats_raw = spark.read.csv(
    f"{BRONZE_PATH}/combats.csv",
    header=True,
    schema=combats_schema,
)

print("pokemon.csv schema:")
df_pokemon_raw.printSchema()

print("combats.csv schema:")
df_combats_raw.printSchema()

pokemon.csv schema:
root
 |-- #: integer (nullable = true)
 |-- Name: string (nullable = true)
 |-- Type 1: string (nullable = true)
 |-- Type 2: string (nullable = true)
 |-- HP: integer (nullable = true)
 |-- Attack: integer (nullable = true)
 |-- Defense: integer (nullable = true)
 |-- Sp. Atk: integer (nullable = true)
 |-- Sp. Def: integer (nullable = true)
 |-- Speed: integer (nullable = true)
 |-- Generation: integer (nullable = true)
 |-- Legendary: boolean (nullable = true)

combats.csv schema:
root
 |-- First_pokemon: integer (nullable = true)
 |-- Second_pokemon: integer (nullable = true)
 |-- Winner: integer (nullable = true)



## Basic validation

Row counts, a preview of the data, null counts per column, and a duplicate check — this is sanity-checking the raw source, not cleaning it. If something looks wrong here, it gets flagged and handled explicitly in Silver, never silently fixed in Bronze.

In [4]:
print(f"pokemon.csv row count: {df_pokemon_raw.count()}")
print(f"combats.csv row count: {df_combats_raw.count()}")

df_pokemon_raw.show(5)
df_combats_raw.show(5)

pokemon.csv row count: 800
combats.csv row count: 50000
+---+-------------+------+------+---+------+-------+-------+-------+-----+----------+---------+
|  #|         Name|Type 1|Type 2| HP|Attack|Defense|Sp. Atk|Sp. Def|Speed|Generation|Legendary|
+---+-------------+------+------+---+------+-------+-------+-------+-----+----------+---------+
|  1|    Bulbasaur| Grass|Poison| 45|    49|     49|     65|     65|   45|         1|    false|
|  2|      Ivysaur| Grass|Poison| 60|    62|     63|     80|     80|   60|         1|    false|
|  3|     Venusaur| Grass|Poison| 80|    82|     83|    100|    100|   80|         1|    false|
|  4|Mega Venusaur| Grass|Poison| 80|   100|    123|    122|    120|   80|         1|    false|
|  5|   Charmander|  Fire|  NULL| 39|    52|     43|     60|     50|   65|         1|    false|
+---+-------------+------+------+---+------+-------+-------+-------+-----+----------+---------+
only showing top 5 rows

+-------------+--------------+------+
|First_pokemon|Se

In [6]:
def null_counts(df, label):
    print(f"--- Null counts: {label} ---")
    df.select([
        F.count(F.when(F.col(f"`{c}`").isNull(), c)).alias(c) for c in df.columns
    ]).show(truncate=False)

null_counts(df_pokemon_raw, "pokemon")
null_counts(df_combats_raw, "combats")

--- Null counts: pokemon ---
+---+----+------+------+---+------+-------+-------+-------+-----+----------+---------+
|#  |Name|Type 1|Type 2|HP |Attack|Defense|Sp. Atk|Sp. Def|Speed|Generation|Legendary|
+---+----+------+------+---+------+-------+-------+-------+-----+----------+---------+
|0  |1   |0     |386   |0  |0     |0      |0      |0      |0    |0         |0        |
+---+----+------+------+---+------+-------+-------+-------+-----+----------+---------+

--- Null counts: combats ---
+-------------+--------------+------+
|First_pokemon|Second_pokemon|Winner|
+-------------+--------------+------+
|0            |0             |0     |
+-------------+--------------+------+



In [7]:
# Duplicate check on primary key candidates
pokemon_dupes = df_pokemon_raw.groupBy("#").count().filter("count > 1")
print(f"Duplicate Pokemon numbers: {pokemon_dupes.count()}")

combats_dupes = df_combats_raw.groupBy("First_pokemon", "Second_pokemon").count().filter("count > 1")
print(f"Duplicate combat pairs (same order): {combats_dupes.count()}")

Duplicate Pokemon numbers: 0
Duplicate combat pairs (same order): 1906


## Note on `Type 2` and `Name` nulls

You'll likely see nulls in `Type 2` (many Pokémon only have one type — that's expected, not a data quality issue) and possibly one null in `Name` (a known quirk in this dataset around Pokémon #63/Primeape depending on the CSV version). We are **not** fixing these here — Bronze just documents what's in the source. Silver decides how to handle them.

## Add ingestion metadata and write to Bronze Parquet

These two columns are lineage/audit metadata, not business transformations — they let us trace every row back to when and from which file it was ingested.

In [8]:
df_pokemon_bronze = (
    df_pokemon_raw
    .withColumn("_ingestion_timestamp", F.current_timestamp())
    .withColumn("_source_file", F.lit("pokemon.csv"))
)

df_combats_bronze = (
    df_combats_raw
    .withColumn("_ingestion_timestamp", F.current_timestamp())
    .withColumn("_source_file", F.lit("combats.csv"))
)

df_pokemon_bronze.write.mode("overwrite").parquet(f"{BRONZE_PATH}/pokemon_parquet")
df_combats_bronze.write.mode("overwrite").parquet(f"{BRONZE_PATH}/combats_parquet")

print("Bronze Parquet written successfully.")

Bronze Parquet written successfully.


## Verify by reading back

In [9]:
check_pokemon = spark.read.parquet(f"{BRONZE_PATH}/pokemon_parquet")
check_combats = spark.read.parquet(f"{BRONZE_PATH}/combats_parquet")

print(f"pokemon_parquet rows: {check_pokemon.count()}")
print(f"combats_parquet rows: {check_combats.count()}")

check_pokemon.show(5)

pokemon_parquet rows: 800
combats_parquet rows: 50000
+---+-------------+------+------+---+------+-------+-------+-------+-----+----------+---------+--------------------+------------+
|  #|         Name|Type 1|Type 2| HP|Attack|Defense|Sp. Atk|Sp. Def|Speed|Generation|Legendary|_ingestion_timestamp|_source_file|
+---+-------------+------+------+---+------+-------+-------+-------+-----+----------+---------+--------------------+------------+
|  1|    Bulbasaur| Grass|Poison| 45|    49|     49|     65|     65|   45|         1|    false|2026-08-11 18:01:...| pokemon.csv|
|  2|      Ivysaur| Grass|Poison| 60|    62|     63|     80|     80|   60|         1|    false|2026-08-11 18:01:...| pokemon.csv|
|  3|     Venusaur| Grass|Poison| 80|    82|     83|    100|    100|   80|         1|    false|2026-08-11 18:01:...| pokemon.csv|
|  4|Mega Venusaur| Grass|Poison| 80|   100|    123|    122|    120|   80|         1|    false|2026-08-11 18:01:...| pokemon.csv|
|  5|   Charmander|  Fire|  NULL| 39

In [10]:
spark.stop()